# AquaVigil: YOLOv8 Baseline Training on Google Colab (GPU)

This notebook runs the full baseline training for **AquaVigil** with GPU acceleration (T4/V100/A100) on Google Colab.

### Pipeline Overview:
1. Verify GPU availability (`torch.cuda.is_available()`)
2. Mount Google Drive
3. Set up directory / clone repository
4. Point to dataset (`dataset/processed/baseline_v1/data.yaml`)
5. Train YOLOv8n (50 epochs, batch 16, imgsz 640)
6. Save checkpoints and evaluation artifacts directly to Google Drive

In [ ]:
# 1. Check GPU Environment
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: No GPU detected! Please enable GPU via Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)")

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Install Requirements
%pip install ultralytics PyYAML

In [ ]:
# 4. Define Paths & Workspace Setup
import os
from pathlib import Path

# Set this path to where your AquaVigil project or dataset folder is stored in your Google Drive
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/AquaVigil")
DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project directory on Drive: {DRIVE_PROJECT_DIR}")

In [ ]:
# 5. Create / Verify Colab data.yaml
# If dataset/processed/baseline_v1 is zipped or copied to Google Drive:
data_yaml_content = f"""
path: {DRIVE_PROJECT_DIR}/dataset/processed/baseline_v1
train: {DRIVE_PROJECT_DIR}/dataset/processed/baseline_v1/train/images
val: {DRIVE_PROJECT_DIR}/dataset/processed/baseline_v1/valid/images
test: {DRIVE_PROJECT_DIR}/dataset/processed/baseline_v1/test/images
nc: 5
names:
  0: Aeromoniasis
  1: branchitis
  2: keratitis
  3: lymphocystis
  4: object
"""

colab_data_yaml = DRIVE_PROJECT_DIR / "colab_data.yaml"
colab_data_yaml.write_text(data_yaml_content.strip())
print(f"Colab data configuration written to {colab_data_yaml}")

In [ ]:
# 6. Train YOLOv8n Baseline Model
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

# Baseline hyperparameter configuration
results = model.train(
    data=str(colab_data_yaml),
    epochs=50,
    batch=16,
    imgsz=640,
    device=0,
    workers=4,
    seed=42,
    patience=15,
    project=str(DRIVE_PROJECT_DIR / "runs"),
    name="baseline_v1_yolov8n",
    exist_ok=True
)

print("Training completed successfully!")
print(f"Best weights saved at: {DRIVE_PROJECT_DIR}/runs/baseline_v1_yolov8n/weights/best.pt")

In [ ]:
# 7. Validate on Validation Split
val_metrics = model.val(
    data=str(colab_data_yaml),
    split='val',
    device=0,
    project=str(DRIVE_PROJECT_DIR / "results"),
    name="eval_val"
)
print("Validation mAP@50:", val_metrics.box.map50)
print("Validation mAP@50-95:", val_metrics.box.map)